In [ ]:

from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler

from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor
)
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from pyspark.ml.evaluation import RegressionEvaluator

from xgboost import XGBRegressor

import pandas as pd
import xgboost
import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

In [ ]:
# ============================================================
# CELL 2 — HCC DATA PREPARATION & FEATURE ENGINEERING
# ============================================================

print("XGBoost version:", xgboost.__version__)


# ============================================================
# 1. LOAD FINAL SILVER DATA
# ============================================================

df_ml_source = spark.table(
    "silver_member_risk_final"
)

print(
    "Source rows:",
    df_ml_source.count()
)


# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================

df_ml = (
    df_ml_source

    .select(
        "member_id",
        "age",
        "gender",
        "chronic_condition_count",
        "hcc_count_v22",
        "payment_hcc_count_v22",
        "risk_score_v22_std",
        "risk_score_v28_std"
    )

    # Gender indicators
    .withColumn(
        "is_male",
        F.when(
            F.col("gender") == "Male",
            1.0
        ).otherwise(0.0)
    )

    .withColumn(
        "is_female",
        F.when(
            F.col("gender") == "Female",
            1.0
        ).otherwise(0.0)
    )

    # ML target
    .withColumn(
        "label",
        F.col("risk_score_v28_std")
        .cast("double")
    )

    # Fill missing values
    .fillna({
        "age": 0,
        "chronic_condition_count": 0,
        "hcc_count_v22": 0,
        "payment_hcc_count_v22": 0,
        "risk_score_v22_std": 0
    })

    # Remove rows without target
    .filter(
        F.col("label").isNotNull()
    )
)


# ============================================================
# 3. FEATURE LIST
# ============================================================

feature_cols = [
    "age",
    "is_male",
    "is_female",
    "chronic_condition_count",
    "hcc_count_v22",
    "payment_hcc_count_v22",
    "risk_score_v22_std"
]

print("Features:")
for feature in feature_cols:
    print(" -", feature)


# ============================================================
# 4. VECTOR ASSEMBLY
# ============================================================

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml_vector = (
    assembler
    .transform(df_ml)
    .select(
        "member_id",
        *feature_cols,
        "features",
        "label"
    )
)


print(
    "ML dataset rows:",
    df_ml_vector.count()
)




In [ ]:
# Stable member-based split: 70 / 15 / 15
split_data = df_ml_vector.withColumn(
    "_bucket",
    F.pmod(
        F.xxhash64(F.col("member_id").cast("string")),
        F.lit(100)
    )
)

train_final = (
    split_data
    .filter(F.col("_bucket") < 70)
    .drop("_bucket")
)

val_final = (
    split_data
    .filter(
        (F.col("_bucket") >= 70) &
        (F.col("_bucket") < 85)
    )
    .drop("_bucket")
)

test_final = (
    split_data
    .filter(F.col("_bucket") >= 85)
    .drop("_bucket")
)

print("Train:", train_final.count())
print("Validation:", val_final.count())
print("Test:", test_final.count())

# Verify isolation
train_ids = train_final.select("member_id")
val_ids = val_final.select("member_id")
test_ids = test_final.select("member_id")

assert train_ids.join(val_ids, "member_id").count() == 0
assert train_ids.join(test_ids, "member_id").count() == 0
assert val_ids.join(test_ids, "member_id").count() == 0

print("PASS: No member overlap across all three sets")

In [ ]:

# ============================================================
# HCC V2 — TRAIN BASELINE REGRESSION MODELS
# Train only on TRAIN; evaluate only on VALIDATION
# ============================================================

from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator

models_final = {
    "Linear Regression": LinearRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=100,
        regParam=0.0
    ),

    "Random Forest": RandomForestRegressor(
        featuresCol="features",
        labelCol="label",
        numTrees=100,
        maxDepth=8,
        seed=42
    )
}

trained_models_final = {}
validation_results = []

for name, estimator in models_final.items():

    print("Training:", name)

    fitted_model = estimator.fit(train_final)

    val_predictions = fitted_model.transform(val_final)

    scores = {}

    for metric in ["mae", "rmse", "r2"]:
        scores[metric] = float(
            RegressionEvaluator(
                labelCol="label",
                predictionCol="prediction",
                metricName=metric
            ).evaluate(val_predictions)
        )

    validation_results.append({
        "model": name,
        **scores
    })

    trained_models_final[name] = fitted_model

print("Baseline model training completed.")
print("Models evaluated:", len(validation_results))
print("XGBoost is trained in the next cell.")


In [ ]:
# ============================================================
# HCC — FIX XGBOOST TRAINING
# Native XGBRegressor: No Spark barrier execution
# ============================================================

# ------------------------------------------------------------
# 1. Verify the other models completed
# ------------------------------------------------------------

completed = {r["model"] for r in validation_results}

assert {
    "Linear Regression",
    "Random Forest"
}.issubset(completed), (
    "Run Linear Regression and Random Forest first."
)

print("Linear Regression: COMPLETE")
print("Random Forest: COMPLETE")

# ------------------------------------------------------------
# 2. Convert only the necessary data to Pandas
# ------------------------------------------------------------

train_pd = (
    train_final
    .select(*feature_cols, "label")
    .toPandas()
)

val_pd = (
    val_final
    .select(*feature_cols, "label")
    .toPandas()
)

X_train = train_pd[feature_cols].astype("float32")
y_train = train_pd["label"].astype("float32")

X_val = val_pd[feature_cols].astype("float32")
y_val = val_pd["label"].astype("float32")

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Features:", len(feature_cols))

# ------------------------------------------------------------
# 3. Train native XGBoost
# ------------------------------------------------------------

xgb_model_final = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    random_state=42,
    n_jobs=4
)

xgb_model_final.fit(X_train, y_train)

print("XGBoost training: COMPLETE")

# ------------------------------------------------------------
# 4. Validation predictions
# ------------------------------------------------------------

xgb_val_pred = xgb_model_final.predict(X_val)

xgb_scores = {
    "model": "XGBoost",
    "mae": float(
        mean_absolute_error(y_val, xgb_val_pred)
    ),
    "rmse": float(
        np.sqrt(mean_squared_error(y_val, xgb_val_pred))
    ),
    "r2": float(
        r2_score(y_val, xgb_val_pred)
    )
}

# ------------------------------------------------------------
# 5. Combine all three validation results
# ------------------------------------------------------------

validation_results_final = [
    r for r in validation_results
    if r["model"] != "XGBoost"
]

validation_results_final.append(xgb_scores)

trained_models_final["XGBoost"] = xgb_model_final

validation_metrics_final = spark.createDataFrame(
    validation_results_final
)

print("\n===== FINAL VALIDATION COMPARISON =====")

display(
    validation_metrics_final.orderBy(F.asc("rmse"))
)

# ------------------------------------------------------------
# 6. Select winning model using validation RMSE
# ------------------------------------------------------------

winner_row = (
    validation_metrics_final
    .orderBy(F.asc("rmse"))
    .first()
)

winner_name = winner_row["model"]
winner_model = trained_models_final[winner_name]

print("\n===== SELECTED MODEL =====")
print("Model:", winner_name)
print("MAE:", round(winner_row["mae"], 4))
print("RMSE:", round(winner_row["rmse"], 4))
print("R2:", round(winner_row["r2"], 4))

print("\nTEST DATASET REMAINS UNTOUCHED")

In [ ]:
print("Target: risk_score_v28_std")
print("Model features:")
for col in feature_cols:
    print(" -", col)

In [ ]:
# ============================================================
# HCC — FINAL INDEPENDENT TEST EVALUATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Validate selected model
assert winner_name == "XGBoost"

# 2. Load test data
# Avoid selecting risk_score_v22_std twice,
# because it is already included in feature_cols.

test_columns = list(dict.fromkeys(
    feature_cols + ["label", "member_id"]
))

test_pd_final = (
    test_final
    .select(*test_columns)
    .toPandas()
)

X_test_final = (
    test_pd_final[feature_cols]
    .astype("float32")
)

y_test_final = (
    test_pd_final["label"]
    .to_numpy(dtype=np.float64)
)

# 3. Predict with trained XGBoost
y_pred_final = winner_model.predict(
    X_test_final
)

# 4. Evaluation function
def evaluate_regression(actual, predicted):
    return {
        "MAE": float(
            mean_absolute_error(actual, predicted)
        ),
        "RMSE": float(
            np.sqrt(mean_squared_error(actual, predicted))
        ),
        "R2": float(
            r2_score(actual, predicted)
        )
    }

# 5. Evaluate selected model
xgb_test = evaluate_regression(
    y_test_final,
    y_pred_final
)

# 6. Compare against V22 baseline
v22_test = evaluate_regression(
    y_test_final,
    test_pd_final["risk_score_v22_std"].to_numpy()
)

# 7. Compare against training-mean baseline
train_mean = train_final.agg(
    {"label": "avg"}
).first()[0]

mean_test = evaluate_regression(
    y_test_final,
    np.full(len(y_test_final), train_mean)
)

# 8. Display comparison
results = pd.DataFrame([
    {"Model": "XGBoost", **xgb_test},
    {"Model": "V22 Baseline", **v22_test},
    {"Model": "Training Mean", **mean_test}
])

print("===== FINAL TEST RESULTS =====")
print("Test members:", len(y_test_final))

display(
    results.sort_values("RMSE")
)

# 9. Improvement over V22
improvement = (
    (v22_test["RMSE"] - xgb_test["RMSE"])
    / v22_test["RMSE"]
) * 100

print(
    f"XGBoost RMSE improvement over V22: "
    f"{improvement:.2f}%"
)

print("\nNo Gold tables modified.")

In [ ]:
# ============================================================
# HCC V22 -> V28 — SAVE VALIDATED MODEL V2
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from datetime import datetime, timezone
from pyspark.sql import functions as F

VERSION = "v2"
RUN_AT = datetime.now(timezone.utc)

MODEL_DIR = "/lakehouse/default/Files/models/hcc_v2"
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Validate existing results
# ------------------------------------------------------------

assert winner_name == "XGBoost"
assert len(y_test_final) == len(y_pred_final)
assert len(y_test_final) == 17288

assert test_pd_final["member_id"].is_unique

print("Validation checks passed.")

# ------------------------------------------------------------
# 2. Save trained XGBoost model
# ------------------------------------------------------------

model_path = (
    f"{MODEL_DIR}/hcc_v28_xgboost_v2.joblib"
)

joblib.dump(winner_model, model_path)

# ------------------------------------------------------------
# 3. Save metadata
# ------------------------------------------------------------

metadata = {
    "project": "CMS-HCC V22 to V28 Risk Transition Simulator",
    "model_version": VERSION,
    "model_name": winner_name,
    "target": "risk_score_v28_std",
    "features": list(feature_cols),
    "train_rows": 81387,
    "validation_rows": 17677,
    "test_rows": 17288,
    "split_method": "deterministic member-level hash",
    "selection_metric": "validation RMSE",
    "final_test_metrics": xgb_test,
    "v22_baseline_metrics": v22_test,
    "training_mean_baseline_metrics": mean_test,
    "created_at": RUN_AT.isoformat()
}

metadata_path = (
    f"{MODEL_DIR}/hcc_v28_xgboost_v2_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

# ------------------------------------------------------------
# 4. Save three-model validation comparison
# ------------------------------------------------------------

validation_pd = validation_metrics_final.toPandas()

validation_pd["model_version"] = VERSION
validation_pd["dataset_split"] = "VALIDATION"
validation_pd["run_at"] = RUN_AT.replace(tzinfo=None)

validation_spark = spark.createDataFrame(validation_pd)

# ------------------------------------------------------------
# 5. Save final test metrics and baselines
# ------------------------------------------------------------

test_metrics_pd = pd.DataFrame([
    {"model": "XGBoost", **xgb_test},
    {"model": "V22 Score Baseline", **v22_test},
    {"model": "Training Mean Baseline", **mean_test}
])

test_metrics_pd.columns = [
    c.lower() for c in test_metrics_pd.columns
]

test_metrics_pd["model_version"] = VERSION
test_metrics_pd["dataset_split"] = "TEST"
test_metrics_pd["run_at"] = RUN_AT.replace(tzinfo=None)

test_metrics_spark = spark.createDataFrame(
    test_metrics_pd
)

# ------------------------------------------------------------
# 6. Save 17,288 test predictions
# ------------------------------------------------------------

predictions_pd = pd.DataFrame({
    "member_id": (
        test_pd_final["member_id"].astype(str).to_numpy()
    ),
    "actual_v28_score": np.asarray(
        y_test_final, dtype=np.float64
    ),
    "predicted_v28_score": np.asarray(
        y_pred_final, dtype=np.float64
    ),
    "v22_score": pd.to_numeric(
        test_pd_final["risk_score_v22_std"]
    ).to_numpy(dtype=np.float64)
})

predictions_pd["prediction_error"] = (
    predictions_pd["actual_v28_score"]
    - predictions_pd["predicted_v28_score"]
)

predictions_pd["absolute_error"] = (
    predictions_pd["prediction_error"].abs()
)

predictions_pd["model_name"] = winner_name
predictions_pd["model_version"] = VERSION
predictions_pd["run_at"] = RUN_AT.replace(tzinfo=None)

predictions_spark = spark.createDataFrame(
    predictions_pd
)

# ------------------------------------------------------------
# 7. Save to versioned Lakehouse Delta tables
# ------------------------------------------------------------

tables = {
    "hcc_v2_validation_metrics": validation_spark,
    "hcc_v2_test_metrics": test_metrics_spark,
    "hcc_v2_test_predictions": predictions_spark
}

for name, df in tables.items():

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(name)
    )

    print(f"Saved: {name} | Rows: {df.count()}")

print("\nMODEL SAVED:", model_path)
print("METADATA SAVED:", metadata_path)

print("\nHCC VALIDATED V2 RESULTS SAVED.")

In [ ]:
# ============================================================
# HCC V2 — PUBLISH VALIDATED RESULTS TO GOLD WAREHOUSE
# ============================================================

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

WAREHOUSE = "Gold_HCC"

gold_tables = {
    "fact_hcc_v2_validation_metrics":
        "hcc_v2_validation_metrics",

    "fact_hcc_v2_test_metrics":
        "hcc_v2_test_metrics",

    "fact_hcc_v2_test_predictions":
        "hcc_v2_test_predictions"
}

for gold_name, silver_name in gold_tables.items():

    target = f"{WAREHOUSE}.dbo.{gold_name}"

    source_df = spark.table(silver_name)

    (
        source_df.write
        .mode("overwrite")
        .synapsesql(target)
    )

    print(
        f"Published {target}: "
        f"{source_df.count()} rows"
    )

print("\nHCC V2 GOLD PUBLISH COMPLETE.")

In [ ]:

# ============================================================
# NB_04 — HCC V22 TO V28 RISK TRANSITION SIMULATOR
# FINAL MODEL EVALUATION & VISUALIZATIONS | VERSION 2
#
# Model: XGBoost Regressor
# Selection: Lowest Validation RMSE
# Evaluation: Independent Held-Out Test Set
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. LOAD VALIDATED V2 ARTIFACTS
# ============================================================

MODEL_DIR = "/lakehouse/default/Files/models/hcc_v2"

MODEL_PATH = f"{MODEL_DIR}/hcc_v28_xgboost_v2.joblib"
METADATA_PATH = (
    f"{MODEL_DIR}/hcc_v28_xgboost_v2_metadata.json"
)

assert os.path.exists(MODEL_PATH), "Saved model not found"
assert os.path.exists(METADATA_PATH), "Metadata not found"

viz_model = joblib.load(MODEL_PATH)

with open(METADATA_PATH, "r") as f:
    viz_metadata = json.load(f)

viz_features = list(viz_metadata["features"])

viz_validation = (
    spark.table("hcc_v2_validation_metrics")
    .toPandas()
)

viz_test_metrics = (
    spark.table("hcc_v2_test_metrics")
    .toPandas()
)

viz_predictions = (
    spark.table("hcc_v2_test_predictions")
    .toPandas()
)


# ============================================================
# 2. VERIFY SAVED ARTIFACTS
# ============================================================

assert viz_metadata["model_name"] == "XGBoost"
assert viz_metadata["model_version"] == "v2"

assert len(viz_validation) == 3
assert len(viz_test_metrics) == 3
assert len(viz_predictions) == 17288

assert viz_predictions["member_id"].is_unique
assert len(viz_features) == len(viz_model.feature_importances_)

viz_validation = (
    viz_validation.sort_values("rmse")
    .reset_index(drop=True)
)

viz_test_metrics = (
    viz_test_metrics.sort_values("rmse")
    .reset_index(drop=True)
)

viz_val_xgb = viz_validation.loc[
    viz_validation["model"] == "XGBoost"
].iloc[0]

# IMPORTANT:
# Do not overwrite 'xgb_test' from the final evaluation cell.

viz_test_xgb = viz_test_metrics.loc[
    viz_test_metrics["model"] == "XGBoost"
].iloc[0]

viz_test_v22 = viz_test_metrics.loc[
    viz_test_metrics["model"] == "V22 Score Baseline"
].iloc[0]

viz_improvement = (
    (viz_test_v22["rmse"] - viz_test_xgb["rmse"])
    / viz_test_v22["rmse"]
) * 100

print("=" * 55)
print("VALIDATED HCC MODEL — V2")
print("=" * 55)
print("Model:", viz_metadata["model_name"])
print("Version:", viz_metadata["model_version"])
print("Target:", viz_metadata["target"])
print("Test members:", len(viz_predictions))
print("Features:", len(viz_features))
print("\nValidation model comparison:")
display(viz_validation[["model", "mae", "rmse", "r2"]])

print("\nFinal test model and baseline comparison:")
display(viz_test_metrics[["model", "mae", "rmse", "r2"]])


# ============================================================
# 3. COMMON PLOTTING HELPERS
# ============================================================

def format_chart(ax, title, ylabel=None):
    ax.set_title(title, fontsize=14, fontweight="bold", pad=14)

    if ylabel:
        ax.set_ylabel(ylabel)

    ax.grid(axis="y", linestyle="--", alpha=0.20)
    ax.set_axisbelow(True)


def label_bars(ax, values, decimals=4):
    for bar, value in zip(ax.patches, values):
        ax.annotate(
            f"{value:.{decimals}f}",
            (
                bar.get_x() + bar.get_width() / 2,
                bar.get_height()
            ),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            fontsize=10,
            fontweight="bold"
        )


# ============================================================
# PLOT 1 — VALIDATION RMSE COMPARISON
#
# Primary model-selection metric.
# Lower RMSE indicates better performance.
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

values = viz_validation["rmse"].to_numpy()

ax.bar(
    viz_validation["model"],
    values
)

label_bars(ax, values)

format_chart(
    ax,
    "Model Comparison — Validation RMSE",
    "RMSE (lower is better)"
)

ax.set_ylim(0, max(values) * 1.16)

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 2 — VALIDATION R-SQUARED COMPARISON
#
# Higher R² indicates a better fit to the target.
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

values = viz_validation["r2"].to_numpy()

ax.bar(
    viz_validation["model"],
    values
)

label_bars(ax, values)

format_chart(
    ax,
    "Model Comparison — Validation R²",
    "R² (higher is better)"
)

ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 3 — FINAL TEST RMSE VS V22 BASELINE
#
# Demonstrates additional value beyond existing V22 scores.
# Training-mean baseline remains in the summary table above.
# ============================================================

baseline_names = [
    "XGBoost V2",
    "V22 Score Baseline"
]

baseline_rmse = [
    float(viz_test_xgb["rmse"]),
    float(viz_test_v22["rmse"])
]

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    baseline_names,
    baseline_rmse
)

label_bars(ax, baseline_rmse)

format_chart(
    ax,
    "Independent Test — XGBoost vs V22 Baseline",
    "Test RMSE (lower is better)"
)

ax.set_ylim(0, max(baseline_rmse) * 1.16)

ax.text(
    0.5,
    0.95,
    f"XGBoost reduces RMSE by {viz_improvement:.2f}%",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=11,
    fontweight="bold"
)

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 4 — ACTUAL VS PREDICTED V28 RISK SCORES
#
# Shows how closely the model estimates observed scores.
# Plot is sampled only for readability.
# All metrics use the full test population.
# ============================================================

viz_sample = viz_predictions.sample(
    n=min(3000, len(viz_predictions)),
    random_state=42
)

actual = viz_sample["actual_v28_score"].to_numpy()
predicted = viz_sample["predicted_v28_score"].to_numpy()

valid = np.isfinite(actual) & np.isfinite(predicted)

actual = actual[valid]
predicted = predicted[valid]

lower = float(min(actual.min(), predicted.min()))
upper = float(max(actual.max(), predicted.max()))

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(
    actual,
    predicted,
    alpha=0.25,
    s=12
)

ax.plot(
    [lower, upper],
    [lower, upper],
    linestyle="--",
    linewidth=2,
    label="Perfect prediction"
)

ax.set_xlim(lower, upper)
ax.set_ylim(lower, upper)
ax.set_aspect("equal", adjustable="box")

ax.set_xlabel("Actual V28 Comparative Risk Score")
ax.set_ylabel("Predicted V28 Comparative Risk Score")

ax.set_title(
    "Actual vs Predicted V28 Risk Scores",
    fontsize=14,
    fontweight="bold"
)

ax.legend()

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 5 — TEST RESIDUAL DISTRIBUTION
#
# Residual = Actual - Predicted
# Values near zero indicate smaller prediction errors.
# ============================================================

viz_residuals = (
    viz_predictions["actual_v28_score"]
    - viz_predictions["predicted_v28_score"]
).dropna()

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    viz_residuals,
    bins=60,
    edgecolor="white",
    alpha=0.85
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=2,
    label="Zero error"
)

ax.axvline(
    viz_residuals.mean(),
    linestyle=":",
    linewidth=2,
    label=f"Mean error = {viz_residuals.mean():.4f}"
)

format_chart(
    ax,
    "Final Test — Prediction Error Distribution",
    "Number of members"
)

ax.set_xlabel("Residual (Actual - Predicted)")
ax.legend()

plt.tight_layout()
plt.show()


# ============================================================
# PLOT 6 — XGBOOST FEATURE IMPORTANCE
#
# Indicates which inputs are most useful to the model.
# This is not a causal explanation.
# ============================================================

viz_importance = pd.DataFrame({
    "feature": viz_features,
    "importance": viz_model.feature_importances_
})

viz_importance = viz_importance.sort_values(
    "importance",
    ascending=True
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(
    viz_importance["feature"],
    viz_importance["importance"]
)

ax.set_title(
    "XGBoost — Feature Importance",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.set_xlabel("Relative Feature Importance")
ax.grid(axis="x", linestyle="--", alpha=0.20)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


# ============================================================
# 4. FINAL PERFORMANCE SUMMARY
# ============================================================

viz_summary = pd.DataFrame([
    {
        "Dataset": "Validation",
        "Model": "XGBoost",
        "MAE": float(viz_val_xgb["mae"]),
        "RMSE": float(viz_val_xgb["rmse"]),
        "R2": float(viz_val_xgb["r2"])
    },
    {
        "Dataset": "Final Test",
        "Model": "XGBoost",
        "MAE": float(viz_test_xgb["mae"]),
        "RMSE": float(viz_test_xgb["rmse"]),
        "R2": float(viz_test_xgb["r2"])
    }
])

print("\n" + "=" * 55)
print("FINAL VALIDATED PERFORMANCE SUMMARY")
print("=" * 55)

display(viz_summary.round(4))

print(f"Test members: {len(viz_predictions):,}")
print(f"V22 baseline RMSE: {viz_test_v22['rmse']:.4f}")
print(f"XGBoost test RMSE: {viz_test_xgb['rmse']:.4f}")
print(f"RMSE improvement: {viz_improvement:.2f}%")

print("\nGenerated 6 model evaluation visualizations.")
print("No model retraining performed.")
print("No Lakehouse or Warehouse tables modified.")


In [ ]:

import numpy as np
import pandas as pd

# Load previously saved test predictions
df = spark.table(
    "hcc_v2_test_predictions"
).toPandas()

# Calculate actual and predicted transitions
df["actual_delta"] = (
    df["actual_v28_score"] - df["v22_score"]
)

df["predicted_delta"] = (
    df["predicted_v28_score"] - df["v22_score"]
)

df["delta_error"] = (
    df["actual_delta"] - df["predicted_delta"]
)

df["absolute_delta_error"] = df["delta_error"].abs()

# Define large transitions using an observed-data threshold
threshold = df["actual_delta"].abs().quantile(0.90)

df["transition_group"] = np.select(
    [
        df["actual_delta"] >= threshold,
        df["actual_delta"] <= -threshold
    ],
    [
        "Large positive transition",
        "Large negative transition"
    ],
    default="Other transitions"
)

summary = (
    df.groupby("transition_group")
    .agg(
        members=("member_id", "count"),
        actual_mean_delta=("actual_delta", "mean"),
        predicted_mean_delta=("predicted_delta", "mean"),
        transition_mae=("absolute_delta_error", "mean"),
        mean_error=("delta_error", "mean")
    )
    .reset_index()
)

display(summary.round(4))

print("Large-transition threshold:", round(threshold, 4))
print(
    "Overall transition MAE:",
    round(df["absolute_delta_error"].mean(), 4)
)


In [ ]:
import numpy as np
import pandas as pd

df = spark.table(
    "hcc_v2_test_predictions"
).toPandas()

df["actual_delta"] = (
    df["actual_v28_score"] - df["v22_score"]
)

df["predicted_delta"] = (
    df["predicted_v28_score"] - df["v22_score"]
)

threshold = df["actual_delta"].abs().quantile(0.90)

df["transition_group"] = np.select(
    [
        df["actual_delta"] >= threshold,
        df["actual_delta"] <= -threshold
    ],
    [
        "Large positive",
        "Large negative"
    ],
    default="Other"
)

df["xgb_abs_error"] = (
    df["actual_v28_score"] -
    df["predicted_v28_score"]
).abs()

df["baseline_abs_error"] = (
    df["actual_v28_score"] -
    df["v22_score"]
).abs()

result = df.groupby("transition_group").agg(
    members=("member_id", "count"),
    xgb_mae=("xgb_abs_error", "mean"),
    v22_baseline_mae=("baseline_abs_error", "mean")
).reset_index()

result["improvement_pct"] = (
    (result["v22_baseline_mae"] - result["xgb_mae"])
    / result["v22_baseline_mae"]
) * 100

display(result.round(4))

In [ ]:
# ================================================================
# CMS-HCC V22 → V28 RISK TRANSITION SIMULATOR
# COMPLETE V2 MODEL DIAGNOSTIC ANALYSIS — SINGLE CELL
#
# Includes:
# 1. Model and prediction validation
# 2. Overall performance comparison
# 3. Positive/negative transition analysis
# 4. Transition-direction accuracy
# 5. Model bias and residual analysis
# 6. Feature importance
# 7. Age/risk cohort evaluation (if features are available)
# 8. Essential visualizations
# 9. Final diagnostic summary
#
# READ ONLY: No retraining, model overwriting or table writes
# ================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ================================================================
# 1. LOAD SAVED V2 ARTIFACTS
# ================================================================

MODEL_DIR = "/lakehouse/default/Files/models/hcc_v2"

MODEL_PATH = f"{MODEL_DIR}/hcc_v28_xgboost_v2.joblib"
METADATA_PATH = f"{MODEL_DIR}/hcc_v28_xgboost_v2_metadata.json"

pred = spark.table(
    "hcc_v2_test_predictions"
).toPandas()

metrics = spark.table(
    "hcc_v2_test_metrics"
).toPandas()

model = None
metadata = {}

if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r") as f:
        metadata = json.load(f)

if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)

required = [
    "member_id",
    "actual_v28_score",
    "predicted_v28_score",
    "v22_score"
]

assert all(c in pred.columns for c in required), \
    "Missing required prediction columns"

assert pred["member_id"].is_unique, \
    "Duplicate member IDs found"

assert len(pred) == 17288, \
    "Unexpected number of test members"

for c in required[1:]:
    pred[c] = pd.to_numeric(pred[c], errors="coerce")

assert np.isfinite(
    pred[required[1:]].to_numpy()
).all(), "Invalid or missing numeric predictions"

print("=" * 70)
print("CMS-HCC V22 → V28 | COMPLETE MODEL DIAGNOSTICS")
print("=" * 70)

print("Test members:", len(pred))
print("Model:", metadata.get("model_name", "XGBoost V2"))
print("Model version:", metadata.get("model_version", "v2"))


# ================================================================
# 2. COMPUTE RISK TRANSITION
# ================================================================

pred["actual_delta"] = (
    pred["actual_v28_score"] -
    pred["v22_score"]
)

pred["predicted_delta"] = (
    pred["predicted_v28_score"] -
    pred["v22_score"]
)

pred["error"] = (
    pred["actual_v28_score"] -
    pred["predicted_v28_score"]
)

pred["abs_error"] = pred["error"].abs()

pred["baseline_error"] = (
    pred["actual_v28_score"] -
    pred["v22_score"]
)

pred["baseline_abs_error"] = (
    pred["baseline_error"].abs()
)


# ================================================================
# 3. OVERALL MODEL PERFORMANCE
# ================================================================

y_true = pred["actual_v28_score"]
y_pred = pred["predicted_v28_score"]
y_base = pred["v22_score"]

comparison = pd.DataFrame([
    {
        "Model": "XGBoost V2",
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    },
    {
        "Model": "V22 Score Baseline",
        "MAE": mean_absolute_error(y_true, y_base),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_base)),
        "R2": r2_score(y_true, y_base)
    }
])

print("\n1. OVERALL PERFORMANCE")
display(comparison.round(4))

xgb_rmse = comparison.loc[
    comparison["Model"] == "XGBoost V2", "RMSE"
].iloc[0]

baseline_rmse = comparison.loc[
    comparison["Model"] == "V22 Score Baseline", "RMSE"
].iloc[0]

rmse_improvement = (
    (baseline_rmse - xgb_rmse) / baseline_rmse
) * 100

print(f"Overall RMSE improvement: {rmse_improvement:.2f}%")


# ================================================================
# 4. TRANSITION COHORTS
# Top 10% of absolute actual V22 → V28 changes
# ================================================================

threshold = float(
    pred["actual_delta"].abs().quantile(0.90)
)

pred["transition_group"] = np.select(
    [
        pred["actual_delta"] >= threshold,
        pred["actual_delta"] <= -threshold
    ],
    [
        "Large positive",
        "Large negative"
    ],
    default="Other"
)

def cohort_metrics(g):
    xgb_mae = g["abs_error"].mean()
    baseline_mae = g["baseline_abs_error"].mean()

    return pd.Series({
        "members": len(g),
        "actual_mean_delta": g["actual_delta"].mean(),
        "predicted_mean_delta": g["predicted_delta"].mean(),
        "xgb_mae": xgb_mae,
        "baseline_mae": baseline_mae,
        "mean_error": g["error"].mean(),
        "improvement_pct": (
            (baseline_mae - xgb_mae) / baseline_mae * 100
            if baseline_mae > 0 else np.nan
        )
    })

cohort = (
    pred.groupby("transition_group")
    .apply(cohort_metrics)
    .reset_index()
)

cohort["members"] = cohort["members"].astype(int)

print("\n2. TRANSITION COHORT ANALYSIS")
print("Large-transition threshold:", round(threshold, 4))

display(cohort.round(4))


# ================================================================
# 5. DIRECTIONAL ACCURACY
#
# Neutral band prevents tiny numerical changes from being
# interpreted as meaningful positive/negative transitions.
# ================================================================

NEUTRAL_BAND = 0.05

def direction(values):
    return np.select(
        [
            values > NEUTRAL_BAND,
            values < -NEUTRAL_BAND
        ],
        [
            "Increase",
            "Decrease"
        ],
        default="Stable"
    )

pred["actual_direction"] = direction(
    pred["actual_delta"]
)

pred["predicted_direction"] = direction(
    pred["predicted_delta"]
)

pred["correct_direction"] = (
    pred["actual_direction"] ==
    pred["predicted_direction"]
)

direction_accuracy = (
    pred["correct_direction"].mean() * 100
)

direction_summary = (
    pred.groupby("actual_direction")
    .agg(
        members=("member_id", "count"),
        direction_accuracy_pct=(
            "correct_direction",
            lambda x: x.mean() * 100
        )
    )
    .reset_index()
)

confusion = pd.crosstab(
    pred["actual_direction"],
    pred["predicted_direction"],
    rownames=["Actual"],
    colnames=["Predicted"]
).reindex(
    index=["Decrease", "Stable", "Increase"],
    columns=["Decrease", "Stable", "Increase"],
    fill_value=0
)

print("\n3. TRANSITION DIRECTION ACCURACY")
print(f"Neutral band: ±{NEUTRAL_BAND}")
print(f"Overall direction accuracy: {direction_accuracy:.2f}%")

display(direction_summary.round(2))
display(confusion)


# ================================================================
# 6. RESIDUAL AND LARGE-ERROR DIAGNOSTICS
# ================================================================

print("\n4. ERROR DIAGNOSTICS")

error_summary = pd.DataFrame([{
    "Mean Error": pred["error"].mean(),
    "Median Absolute Error": pred["abs_error"].median(),
    "90th Percentile Absolute Error": pred["abs_error"].quantile(0.90),
    "95th Percentile Absolute Error": pred["abs_error"].quantile(0.95),
    "99th Percentile Absolute Error": pred["abs_error"].quantile(0.99),
    "Maximum Absolute Error": pred["abs_error"].max()
}])

display(error_summary.round(4))

print("\nLargest prediction errors:")

largest_errors = pred.nlargest(
    10, "abs_error"
)[[
    "member_id",
    "v22_score",
    "actual_v28_score",
    "predicted_v28_score",
    "actual_delta",
    "predicted_delta",
    "abs_error"
]]

display(largest_errors.round(4))


# ================================================================
# 7. FEATURE IMPORTANCE
# ================================================================

importance = None

if model is not None and hasattr(model, "feature_importances_"):

    features = metadata.get("features", [])

    if len(features) == len(model.feature_importances_):

        importance = pd.DataFrame({
            "feature": features,
            "importance": model.feature_importances_
        }).sort_values(
            "importance", ascending=False
        )

        print("\n5. XGBOOST FEATURE IMPORTANCE")
        display(importance.round(4))

    else:
        print("\nFeature importance skipped: metadata mismatch.")

else:
    print("\nFeature importance unavailable: model not loaded.")


# ================================================================
# 8. OPTIONAL COHORT ANALYSIS
#
# Prediction table may only contain member_id and score fields.
# Do not guess upstream feature values or create fake joins.
# ================================================================

print("\n6. OPTIONAL AGE / RISK-TIER DIAGNOSTICS")

available_cohorts = [
    c for c in ["age_band", "risk_tier"]
    if c in pred.columns
]

if available_cohorts:

    for col in available_cohorts:

        cohort_result = (
            pred.groupby(col, dropna=False)
            .agg(
                members=("member_id", "count"),
                xgb_mae=("abs_error", "mean"),
                baseline_mae=("baseline_abs_error", "mean")
            )
            .reset_index()
        )

        print(f"\n{col} performance:")
        display(cohort_result.round(4))

else:
    print(
        "Age/risk-tier diagnostics skipped because these "
        "columns are not in the saved prediction table. "
        "Join to a verified upstream member table before "
        "performing demographic cohort analysis."
    )


# ================================================================
# 9. ESSENTIAL MODEL VISUALIZATIONS
# ================================================================

# CHART 1 — Actual vs predicted mean transitions
plot_cohort = cohort.set_index("transition_group").reindex(
    ["Large negative", "Large positive", "Other"]
)

ax = plot_cohort[[
    "actual_mean_delta",
    "predicted_mean_delta"
]].plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Actual vs Predicted V22 → V28 Transitions")
ax.set_ylabel("Mean Risk Score Change")
ax.set_xlabel("")
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis="x", rotation=0)
ax.legend(["Actual", "Predicted"])
plt.tight_layout()
plt.show()


# CHART 2 — XGBoost vs baseline MAE by cohort
ax = plot_cohort[[
    "xgb_mae",
    "baseline_mae"
]].plot(
    kind="bar",
    figsize=(10, 5)
)

ax.set_title("Cohort MAE — XGBoost vs V22 Baseline")
ax.set_ylabel("MAE (Lower is Better)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(["XGBoost", "V22 Baseline"])
plt.tight_layout()
plt.show()


# CHART 3 — Actual vs predicted risk scores
sample = pred.sample(
    n=min(3000, len(pred)),
    random_state=42
)

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(
    sample["actual_v28_score"],
    sample["predicted_v28_score"],
    alpha=0.25,
    s=12
)

limits = [
    min(
        sample["actual_v28_score"].min(),
        sample["predicted_v28_score"].min()
    ),
    max(
        sample["actual_v28_score"].max(),
        sample["predicted_v28_score"].max()
    )
]

ax.plot(limits, limits, "--", label="Perfect prediction")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Actual V28 Score")
ax.set_ylabel("Predicted V28 Score")
ax.set_title("Actual vs Predicted V28 Risk Scores")
ax.legend()
plt.tight_layout()
plt.show()


# CHART 4 — Residual distribution
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    pred["error"],
    bins=60,
    edgecolor="white"
)

ax.axvline(0, linestyle="--", label="Zero error")
ax.axvline(
    pred["error"].mean(),
    linestyle=":",
    label="Mean error"
)

ax.set_title("XGBoost Prediction Error Distribution")
ax.set_xlabel("Actual - Predicted")
ax.set_ylabel("Members")
ax.legend()
plt.tight_layout()
plt.show()


# CHART 5 — Feature importance
if importance is not None:

    ax = importance.sort_values(
        "importance"
    ).plot(
        x="feature",
        y="importance",
        kind="barh",
        legend=False,
        figsize=(9, 5)
    )

    ax.set_title("XGBoost Feature Importance")
    ax.set_xlabel("Relative Importance")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()


# ================================================================
# 10. FINAL DIAGNOSTIC SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("FINAL HCC V2 DIAGNOSTIC SUMMARY")
print("=" * 70)

print(f"Test members: {len(pred):,}")
print(f"XGBoost RMSE: {xgb_rmse:.4f}")
print(f"V22 baseline RMSE: {baseline_rmse:.4f}")
print(f"Overall RMSE improvement: {rmse_improvement:.2f}%")
print(f"Transition direction accuracy: {direction_accuracy:.2f}%")
print(f"Overall MAE: {pred['abs_error'].mean():.4f}")
print(f"Large transition threshold: {threshold:.4f}")

negative = cohort[
    cohort["transition_group"] == "Large negative"
]

positive = cohort[
    cohort["transition_group"] == "Large positive"
]

if not negative.empty:
    print(
        "Large-negative improvement:",
        f"{negative['improvement_pct'].iloc[0]:.2f}%"
    )

if not positive.empty:
    print(
        "Large-positive improvement:",
        f"{positive['improvement_pct'].iloc[0]:.2f}%"
    )

print("\nINTERPRETATION")

if rmse_improvement > 0:
    print("PASS: XGBoost outperforms V22 baseline overall.")
else:
    print("WARNING: XGBoost does not outperform baseline overall.")

if not negative.empty and negative["improvement_pct"].iloc[0] < 0:
    print(
        "WARNING: XGBoost underperforms V22 baseline "
        "for large negative transitions."
    )

if not positive.empty and positive["improvement_pct"].iloc[0] > 0:
    print(
        "PASS: XGBoost improves large positive "
        "transition estimation."
    )

print(
    "\nNEXT: Verify upstream feature provenance, inspect "
    "large-negative transition errors, and evaluate whether "
    "additional V22 diagnosis-level features are available."
)

print("\nAnalysis complete. No models or tables modified.")

In [ ]:
print("Default Lakehouse database:")
print(spark.catalog.currentDatabase())

for table in [
    "hcc_v2_validation_metrics",
    "hcc_v2_test_metrics",
    "hcc_v2_test_predictions"
]:
    print(f"{table}: {spark.table(table).count():,} rows")

In [ ]:

# ============================================================
# NB_04 — PUBLISH LATEST HCC V2 DIAGNOSTICS TO GOLD
# Source: Silver_HCC (default pinned Lakehouse)
# Target: Gold_HCC Warehouse
#
# No model retraining
# Existing Gold V2 tables remain unchanged
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime, timezone

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

WAREHOUSE = "Gold_HCC"
VERSION = "v2"
RUN_AT = datetime.now(timezone.utc).replace(tzinfo=None)

# ------------------------------------------------------------
# 1. LOAD AND VALIDATE SAVED V2 PREDICTIONS
# ------------------------------------------------------------

df = spark.table(
    "hcc_v2_test_predictions"
).toPandas()

required = [
    "member_id",
    "actual_v28_score",
    "predicted_v28_score",
    "v22_score"
]

assert all(c in df.columns for c in required)
assert len(df) == 17288
assert df["member_id"].is_unique

for c in required[1:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

assert np.isfinite(df[required[1:]].to_numpy()).all()

print("Source validation passed:", len(df), "members")

# ------------------------------------------------------------
# 2. TRANSITION FEATURES AND ERRORS
# ------------------------------------------------------------

df["actual_delta"] = (
    df["actual_v28_score"] - df["v22_score"]
)

df["predicted_delta"] = (
    df["predicted_v28_score"] - df["v22_score"]
)

df["prediction_error"] = (
    df["actual_v28_score"] - df["predicted_v28_score"]
)

df["absolute_error"] = df["prediction_error"].abs()

df["baseline_absolute_error"] = df["actual_delta"].abs()

threshold = float(df["actual_delta"].abs().quantile(0.90))

df["transition_group"] = np.select(
    [
        df["actual_delta"] >= threshold,
        df["actual_delta"] <= -threshold
    ],
    ["Large positive", "Large negative"],
    default="Other"
)

NEUTRAL_BAND = 0.05

def classify_direction(values):
    return np.select(
        [
            values > NEUTRAL_BAND,
            values < -NEUTRAL_BAND
        ],
        ["Increase", "Decrease"],
        default="Stable"
    )

df["actual_direction"] = classify_direction(
    df["actual_delta"]
)

df["predicted_direction"] = classify_direction(
    df["predicted_delta"]
)

df["correct_direction"] = (
    df["actual_direction"] == df["predicted_direction"]
).astype(int)

df["model_version"] = VERSION
df["run_at"] = RUN_AT

# ------------------------------------------------------------
# 3. MEMBER-LEVEL TRANSITION PREDICTIONS
# ------------------------------------------------------------

detail_cols = [
    "member_id",
    "v22_score",
    "actual_v28_score",
    "predicted_v28_score",
    "actual_delta",
    "predicted_delta",
    "prediction_error",
    "absolute_error",
    "baseline_absolute_error",
    "transition_group",
    "actual_direction",
    "predicted_direction",
    "correct_direction",
    "model_version",
    "run_at"
]

detail_pd = df[detail_cols].copy()
detail_pd["member_id"] = detail_pd["member_id"].astype(str)

# ------------------------------------------------------------
# 4. TRANSITION COHORT PERFORMANCE
# ------------------------------------------------------------

cohort_pd = (
    df.groupby("transition_group")
    .agg(
        members=("member_id", "count"),
        actual_mean_delta=("actual_delta", "mean"),
        predicted_mean_delta=("predicted_delta", "mean"),
        xgb_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_absolute_error", "mean"),
        mean_error=("prediction_error", "mean"),
        direction_accuracy=(
            "correct_direction", "mean"
        )
    )
    .reset_index()
)

cohort_pd["improvement_pct"] = (
    (
        cohort_pd["baseline_mae"] -
        cohort_pd["xgb_mae"]
    ) / cohort_pd["baseline_mae"]
) * 100

cohort_pd["direction_accuracy_pct"] = (
    cohort_pd["direction_accuracy"] * 100
)

cohort_pd = cohort_pd.drop(
    columns=["direction_accuracy"]
)

cohort_pd["model_version"] = VERSION
cohort_pd["run_at"] = RUN_AT

# ------------------------------------------------------------
# 5. DIRECTION CONFUSION MATRIX
# ------------------------------------------------------------

directions = ["Decrease", "Stable", "Increase"]

confusion_pd = (
    pd.crosstab(
        df["actual_direction"],
        df["predicted_direction"]
    )
    .reindex(
        index=directions,
        columns=directions,
        fill_value=0
    )
    .rename_axis(
        index="actual_direction",
        columns="predicted_direction"
    )
    .stack()
    .rename("members")
    .reset_index()
)

confusion_pd["members"] = (
    confusion_pd["members"].astype(int)
)

confusion_pd["model_version"] = VERSION
confusion_pd["run_at"] = RUN_AT

# ------------------------------------------------------------
# 6. OVERALL DIAGNOSTIC SUMMARY
# ------------------------------------------------------------

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

actual = df["actual_v28_score"]
predicted = df["predicted_v28_score"]
baseline = df["v22_score"]

xgb_mae = mean_absolute_error(actual, predicted)
xgb_rmse = np.sqrt(
    mean_squared_error(actual, predicted)
)
xgb_r2 = r2_score(actual, predicted)

baseline_mae = mean_absolute_error(actual, baseline)
baseline_rmse = np.sqrt(
    mean_squared_error(actual, baseline)
)

summary_pd = pd.DataFrame([{
    "model_version": VERSION,
    "test_members": int(len(df)),
    "xgb_mae": float(xgb_mae),
    "xgb_rmse": float(xgb_rmse),
    "xgb_r2": float(xgb_r2),
    "baseline_mae": float(baseline_mae),
    "baseline_rmse": float(baseline_rmse),
    "rmse_improvement_pct": float(
        (baseline_rmse - xgb_rmse) /
        baseline_rmse * 100
    ),
    "direction_accuracy_pct": float(
        df["correct_direction"].mean() * 100
    ),
    "large_transition_threshold": threshold,
    "neutral_direction_band": NEUTRAL_BAND,
    "run_at": RUN_AT
}])

# ------------------------------------------------------------
# 7. PRE-PUBLISH VALIDATION
# ------------------------------------------------------------

assert len(detail_pd) == 17288
assert len(cohort_pd) == 3
assert len(confusion_pd) == 9
assert confusion_pd["members"].sum() == 17288
assert len(summary_pd) == 1

assert abs(xgb_mae - 0.3269) < 0.001
assert abs(xgb_rmse - 0.6925) < 0.001

print("\nPRE-PUBLISH VALIDATION PASSED")
print("Detail rows:", len(detail_pd))
print("Cohort rows:", len(cohort_pd))
print("Direction rows:", len(confusion_pd))
print("Summary rows:", len(summary_pd))

# ------------------------------------------------------------
# 8. PUBLISH TO GOLD_HCC WAREHOUSE
# ------------------------------------------------------------

gold_tables = {
    "fact_hcc_v2_transition_predictions": detail_pd,
    "fact_hcc_v2_transition_analysis": cohort_pd,
    "fact_hcc_v2_direction_metrics": confusion_pd,
    "fact_hcc_v2_diagnostic_summary": summary_pd
}

for table_name, pandas_df in gold_tables.items():

    target = f"{WAREHOUSE}.dbo.{table_name}"

    source_df = spark.createDataFrame(pandas_df)

    (
        source_df.write
        .mode("overwrite")
        .synapsesql(target)
    )

    print(
        f"Published: {target} "
        f"| Rows: {source_df.count():,}"
    )

# ------------------------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("HCC V2 GOLD DIAGNOSTIC PUBLISH COMPLETE")
print("=" * 60)

print(f"XGBoost R2: {xgb_r2:.4f}")
print(f"XGBoost MAE: {xgb_mae:.4f}")
print(f"XGBoost RMSE: {xgb_rmse:.4f}")
print(
    "RMSE improvement:",
    f"{summary_pd['rmse_improvement_pct'].iloc[0]:.2f}%"
)
print(
    "Direction accuracy:",
    f"{summary_pd['direction_accuracy_pct'].iloc[0]:.2f}%"
)

print("\nTransition cohorts:")
print(cohort_pd.round(4).to_string(index=False))

print("\nAll four diagnostic tables published to Gold_HCC.")
print("Existing V2 model and Gold tables were not changed.")


# Conclusion — CMS-HCC V22 to V28 Risk Transition Simulator

## 1. Project Objective and Outcome

The objective of this project was to develop an end-to-end healthcare analytics and machine learning platform that analyzes changes in standardized Medicare risk scores between CMS-HCC V22 and V28.

The solution integrates **Microsoft Fabric Medallion Architecture (Bronze → Silver → Gold)** with comparative HCC risk scoring, regression modeling, and analytical reporting.

**Outcome:** The project successfully implemented the comparative risk analysis and demonstrated that machine learning can estimate standardized V28 risk scores using V22-based risk indicators and member characteristics.

## 2. Machine Learning Approach

The experiment used **116,352 unique member records** and seven predictive features, including age, gender indicators, chronic-condition count, V22 HCC counts, and standardized V22 risk scores.

The dataset was divided using a deterministic, member-isolated split:

| Dataset | Members | Purpose |
|---|---:|---|
| Training | 81,387 | Model training |
| Validation | 17,677 | Model comparison and selection |
| Test | 17,288 | Independent final evaluation |

Three regression algorithms were evaluated:

- **Linear Regression:** Established an interpretable linear benchmark.
- **Random Forest Regressor:** Captured nonlinear relationships and feature interactions using an ensemble of decision trees.
- **XGBoost Regressor:** Used gradient boosting to model complex relationships and reduce prediction errors.

The winning model was selected using the **lowest validation RMSE**, without using the final test dataset for model selection.

## 3. Model Comparison — Validation Results

| Model | MAE ↓ | RMSE ↓ | R² ↑ |
|---|---:|---:|---:|
| **XGBoost** | **0.3249** | **0.6775** | **0.9087** |
| Linear Regression | 0.3401 | 0.6847 | 0.9067 |
| Random Forest | 0.3409 | 0.7017 | 0.9020 |

**Model Selection:** XGBoost achieved the lowest MAE and RMSE and the highest R² on validation data. RMSE was the primary selection criterion because it penalizes larger prediction errors more heavily.

Its advantage over Linear Regression was modest, but consistent across all three validation metrics.

## 4. Final Independent Test Results

The selected XGBoost model was evaluated on **17,288 previously unseen members**.

| Model | Test MAE ↓ | Test RMSE ↓ | Test R² ↑ |
|---|---:|---:|---:|
| **XGBoost V2** | **0.3269** | **0.6925** | **0.9085** |
| V22 Score Baseline | 0.3465 | 0.7207 | 0.9009 |
| Training Mean Baseline | 1.6912 | 2.2889 | Approximately 0 |

### Key Findings

- **R² = 0.9085:** The model explains approximately 90.85% of the variation in standardized V28 scores within the held-out test population.
- **MAE = 0.3269:** The average absolute prediction error was approximately 0.327 standardized risk-score units.
- **RMSE = 0.6925:** The model achieved lower overall prediction error than both baseline methods.
- **3.90% RMSE improvement:** XGBoost reduced test RMSE by approximately 3.90% compared with directly using the standardized V22 risk score.

The validation and final test R² values were nearly identical (**0.9087 versus 0.9085**), providing encouraging evidence of performance stability across held-out members.

## 5. Production-Style Implementation

The solution includes:

- Bronze ingestion and Silver data transformation.
- Standardized V22/V28 risk-score comparison.
- Gold Warehouse dimensional modeling.
- Three-model regression comparison.
- Member-isolated training, validation, and testing.
- Versioned XGBoost model persistence.
- Saved validation metrics, test metrics, and member-level predictions.
- Model performance visualizations, feature importance, and prediction-error analysis.
- Microsoft Fabric integration for downstream analytical reporting.

The validated experiment and model artifacts are retained as **Version 2 (v2)**, preserving earlier outputs for comparison and traceability.

## 6. Limitations and Future Improvements

The results apply to a **comparative standardized risk-score estimation task**, not to future clinical deterioration prediction or official Medicare payment determination.

The V22 score is already strongly associated with V28 scores; therefore, the improvement from XGBoost over the V22 baseline is meaningful but relatively modest.

Additional validation should confirm the complete upstream feature provenance, prevent the inclusion of information unavailable at the intended scoring time, and examine performance across member risk tiers and demographic groups. External validation would also be needed to establish generalizability beyond the current dataset.

## 7. Final Conclusion

**The CMS-HCC V22 to V28 Risk Transition Simulator successfully achieved its technical and analytical objectives.**

XGBoost was selected through validation-based model comparison and achieved **MAE 0.3269, RMSE 0.6925, and R² 0.9085** on an independent 17,288-member test set.

The model demonstrated stable held-out performance and delivered a **3.90% reduction in RMSE over the V22 score baseline**.

The project provides a reproducible, production-style foundation for standardized HCC risk-transition analysis, with versioned model evaluation, warehouse reporting, and traceable predictions.

**Project Status: End-to-end implementation completed; internal ML evaluation validated, with upstream feature-provenance and external generalization checks remaining.**
